In [1]:
import os
import random
from datetime import datetime, timedelta
from typing import List, Tuple

import cv2
import numpy as np
import qrcode
from PIL import Image

In [2]:
DATASET_DIR = "dataset9"
IMAGES_DIR = os.path.join(DATASET_DIR, "images")
LABELS_DIR = os.path.join(DATASET_DIR, "labels")

QR_SIZE = 21
PATCH_SIZE = 10

DATASET_SIZE = 20_000
IMAGE_SIZE = QR_SIZE * PATCH_SIZE

START_DATETIME = datetime(2023, 1, 1, 0, 0, 0)
END_DATETIME = datetime(2026, 12, 31, 23, 59, 59)

In [3]:
os.makedirs(IMAGES_DIR, exist_ok=True)
os.makedirs(LABELS_DIR, exist_ok=True)

In [4]:
def get_random_datetime(start: datetime, end: datetime) -> datetime:
    total_seconds = int((end - start).total_seconds())
    return start + timedelta(seconds=random.randint(0, total_seconds))


def build_qr_code(payload: str) -> Tuple[List[List[bool]], np.ndarray]:
    qr = qrcode.QRCode(
        version=1,
        error_correction=qrcode.constants.ERROR_CORRECT_M,
        box_size=1,
        border=0,
    )
    qr.add_data(payload)
    qr.make(fit=False)

    matrix = qr.get_matrix()
    base = Image.new("1", (QR_SIZE, QR_SIZE), 1)
    pixel_access = base.load()
    
    for y in range(QR_SIZE):
        for x in range(QR_SIZE):
            pixel_access[x, y] = 0 if matrix[y][x] else 1

    image = base.resize((IMAGE_SIZE, IMAGE_SIZE), Image.NEAREST).convert("L")
    return matrix, np.array(image)

In [5]:
def apply_padding(image: np.ndarray) -> np.ndarray:
    height, width = image.shape

    min_ratio = -0.04
    max_ratio = 0.09

    padding_left = int(round(random.uniform(min_ratio, max_ratio) * width))
    padding_right = int(round(random.uniform(min_ratio, max_ratio) * width))
    padding_top = int(round(random.uniform(min_ratio, max_ratio) * height))
    padding_bottom = int(round(random.uniform(min_ratio, max_ratio) * height))

    crop_top = max(0, -padding_top)
    crop_bottom = max(0, -padding_bottom)
    crop_left = max(0, -padding_left)
    crop_right = max(0, -padding_right)

    if crop_top or crop_bottom or crop_left or crop_right:
        image = image[
            crop_top : height - crop_bottom,
            crop_left : width - crop_right,
        ]

    border_top = max(0, padding_top)
    border_bottom = max(0, padding_bottom)
    border_left = max(0, padding_left)
    border_right = max(0, padding_right)

    if border_top or border_bottom or border_left or border_right:
        image = cv2.copyMakeBorder(
            image,
            border_top,
            border_bottom,
            border_left,
            border_right,
            borderType=cv2.BORDER_CONSTANT,
            value=255,
        )

    return cv2.resize(
        image,
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=cv2.INTER_LINEAR,
    )


def apply_morphology(image: np.ndarray) -> np.ndarray:
    if random.random() >= 0.25:
        return image

    kernel = np.ones((2, 2), np.uint8)

    operation = random.choice([cv2.MORPH_ERODE, cv2.MORPH_DILATE])
    return cv2.morphologyEx(image, operation, kernel, iterations=1)


def apply_curl(image: np.ndarray) -> np.ndarray:
    if random.random() >= 0.3:
        return image

    height, width = image.shape

    amplitude = random.uniform(1.0, 2.5)
    frequency = random.uniform(0.6, 1.2)
    phase = random.uniform(0, 2 * np.pi)

    x, y = np.meshgrid(np.arange(width), np.arange(height))

    offset_x = amplitude * np.sin(
        2 * np.pi * frequency * y / height + phase
    )

    map_x = (x + offset_x).astype(np.float32)
    map_y = y.astype(np.float32)

    return cv2.remap(
        image,
        map_x,
        map_y,
        interpolation=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=255,
    )


def apply_perspective(image: np.ndarray) -> np.ndarray:
    height, width = image.shape

    margin = random.uniform(0.015, 0.035) * width
    src = np.float32([
        [0, 0],
        [width, 0],
        [width, height],
        [0, height],
    ])
    dst = np.float32([
        [random.uniform(0, margin), random.uniform(0, margin)],
        [width - random.uniform(0, margin), random.uniform(0, margin)],
        [width - random.uniform(0, margin), height - random.uniform(0, margin)],
        [random.uniform(0, margin), height - random.uniform(0, margin)],
    ])

    matrix = cv2.getPerspectiveTransform(src, dst)
    return cv2.warpPerspective(
        image,
        matrix,
        (width, height),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=255,
    )


def apply_anisotropic_scale(image: np.ndarray) -> np.ndarray:
    scale_x = random.uniform(0.97, 1.03)
    scale_y = random.uniform(0.97, 1.03)

    image = cv2.resize(
        image,
        None,
        fx=scale_x,
        fy=scale_y,
        interpolation=cv2.INTER_LINEAR,
    )

    return cv2.resize(
        image,
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=cv2.INTER_LINEAR,
    )


def apply_gray(image: np.ndarray) -> np.ndarray:
    gray_level = random.randint(175, 200)
    texture = np.random.normal(0, 2.5, image.shape)
    paper = np.full(image.shape, gray_level, dtype=np.float32) + texture

    out = np.where(
        image > 128,
        paper,
        image.astype(np.float32),
    )
    return np.clip(out, 0, 255).astype(np.uint8)


def apply_black_lift(image: np.ndarray) -> np.ndarray:
    black_level = random.randint(35, 65)
    dark_noise = np.random.normal(0, 3.0, image.shape)
    lifted_black = black_level + dark_noise

    out = np.where(
        image < 128,
        lifted_black,
        image.astype(np.float32),
    )
    return np.clip(out, 0, 255).astype(np.uint8)


def apply_lighting(image: np.ndarray) -> np.ndarray:
    height, width = image.shape

    center_x = random.uniform(-0.3 * width, 1.3 * width)
    center_y = random.uniform(-0.3 * height, 1.3 * height)
    yy, xx = np.meshgrid(np.arange(height), np.arange(width), indexing="ij")

    distance = np.sqrt((xx - center_x) ** 2 + (yy - center_y) ** 2)
    distance = distance / distance.max()

    strength = random.uniform(0.15, 0.35)
    mask = 1.0 - strength * distance

    out = image.astype(np.float32) * mask
    return np.clip(out, 0, 255).astype(np.uint8)


def apply_blur(image: np.ndarray) -> np.ndarray:
    if random.random() >= 0.9:
        return image

    kernel = random.choice([(3, 3), (3, 3), (5, 5)])
    sigma = random.uniform(0.6, 1.4)
    return cv2.GaussianBlur(image, kernel, sigmaX=sigma)


def apply_contrast(image: np.ndarray) -> np.ndarray:
    alpha = random.uniform(0.9, 1.1)
    beta = random.uniform(-8, 8)

    return cv2.convertScaleAbs(image, alpha=alpha, beta=beta)


def apply_noise(image: np.ndarray) -> np.ndarray:
    if random.random() >= 0.6:
        return image

    sigma = random.uniform(1.5, 4.5)
    noise = np.random.normal(0, sigma, image.shape)

    out = image.astype(np.float32) + noise
    return np.clip(out, 0, 255).astype(np.uint8)


def apply_jpeg(image: np.ndarray) -> np.ndarray:
    if random.random() >= 0.7:
        return image

    quality = random.randint(55, 90)
    _, encoded = cv2.imencode(".jpg", image, [cv2.IMWRITE_JPEG_QUALITY, quality])
    return cv2.imdecode(encoded, cv2.IMREAD_GRAYSCALE)


def augment_image(image: np.ndarray) -> np.ndarray:
    image = apply_padding(image)
    image = apply_morphology(image)
    image = apply_curl(image)
    image = apply_perspective(image)
    image = apply_anisotropic_scale(image)

    image = apply_gray(image)
    image = apply_black_lift(image)

    image = apply_lighting(image)
    image = apply_blur(image)
    image = apply_contrast(image)
    image = apply_noise(image)
    image = apply_jpeg(image)

    return image

In [6]:
def write_label(path: str, matrix: List[List[bool]]) -> None:
    with open(path, "w", encoding="utf-8") as file:
        for row in matrix:
            file.write(" ".join("1" if v else "0" for v in row) + "\n")

In [7]:
used_names = set()
for i in range(DATASET_SIZE):
    while True:
        datetime = get_random_datetime(START_DATETIME, END_DATETIME)
        basename = datetime.strftime("%d%m%Y%H%M%S")
        if basename not in used_names:
            used_names.add(basename)
            break

    payload = datetime.strftime("%d-%m-%Y %H:%M:%S")

    matrix, qr_code = build_qr_code(payload)
    augmented_qr_code = augment_image(qr_code)

    image_path = os.path.join(IMAGES_DIR, f"{basename}.png")
    label_path = os.path.join(LABELS_DIR, f"{basename}.txt")
    
    cv2.imwrite(image_path, augmented_qr_code)
    write_label(label_path, matrix)

    if (i+1) % 500 == 0:
        print(f"{i+1}/{DATASET_SIZE} images generated.")

500/20000 images generated.
1000/20000 images generated.
1500/20000 images generated.
2000/20000 images generated.
2500/20000 images generated.
3000/20000 images generated.
3500/20000 images generated.
4000/20000 images generated.
4500/20000 images generated.
5000/20000 images generated.
5500/20000 images generated.
6000/20000 images generated.
6500/20000 images generated.
7000/20000 images generated.
7500/20000 images generated.
8000/20000 images generated.
8500/20000 images generated.
9000/20000 images generated.
9500/20000 images generated.
10000/20000 images generated.
10500/20000 images generated.
11000/20000 images generated.
11500/20000 images generated.
12000/20000 images generated.
12500/20000 images generated.
13000/20000 images generated.
13500/20000 images generated.
14000/20000 images generated.
14500/20000 images generated.
15000/20000 images generated.
15500/20000 images generated.
16000/20000 images generated.
16500/20000 images generated.
17000/20000 images generated.
